# SERSEM Stage 1 — original LUMIA, reconstructed grey-box, experimental fusion

Reference: Dikici et al., [SERSEM (2026), Sections 2.1/2.2 and Figure 1](https://arxiv.org/pdf/2604.01147). Original [author repository](https://github.com/Serdark4ra/SERSEM-Poisoned-Chalice-Competition-2026), pinned at `413f56040e5b4805bcf15ed794dec56bc4e16b41`.

**Unexecuted notebook: no published result is reproduced or claimed.** Three separately labelled branches are available:

- `lumia`: unchanged upstream hidden-activation pooling, MLP training/selection and top-five arithmetic ensemble. Its language omission and stopping-snapshot behavior are preserved.
- `greybox`: **our Section 2.1 reconstruction**, using raw output-logit Z-scores, sigmoid and structurally weighted pooling. It explicitly forwards language to AST/dictionary/Python linting and never uses hidden states or probes.
- `fusion`: **our experiment**, not a recovered author rule: `alpha * lumia + (1-alpha) * greybox`. A fixed alpha is default; optional selection uses a separate fusion-development pool.

Read `AUDIT.md` for equations, all reconstruction assumptions and the author-artifact search. Accessible Git history contains no grey-box class or cross-branch fusion. Figure 1 shows probe ensembling, not a cross-branch fusion formula. Figshare's cited supplementary endpoints returned HTTP 403 and remain **unverified**, so we cannot assert that no unpublished-to-us implementation exists there. The HF repository lists both 3B and 7B probe pickles but no grey-box source; weights were not downloaded.

Enable Kaggle GPU and Internet, start a fresh session and run cells manually in order. Full 8192-token mode may exceed Kaggle VRAM, RAM, disk or runtime limits. No CPU inference, quantization, automatic shorter context or random-score fallback is supported.

The user reports that the earlier three-branch smoke test succeeded. **This new bounded-resource path has only been statically checked**, not run on T4 here. Default mode is an intermediate resource-validation experiment; it does not evaluate held-out samples by default. Full means the configured 25,000-shadow/25,000-test experiment, not every official training row.


**Immediate research workflow:** timing-only is enabled by default. No competition submission or adaptive work is included. Run the timing pass, then set TIMING_ONLY=False. For the fastest standalone LUMIA result choose BRANCHES=["lumia"] in a new run (or preserve the original settings to reuse an existing valid run). Unverified published probes are not a safe shortcut. See RESEARCH_RUN.md in the source package.


In [ ]:
# 1. User settings: edit this cell, then run all cells in order.
from pathlib import Path
SOURCE_MODE = "uploaded"  # "uploaded" or "github"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
# This mount matches your Kaggle export; change it for a different Dataset slug.
UPLOAD_SOURCE = INPUT_ROOT / "datasets/sayonsujitmondal/sersem-kaggle-source3"
UPLOAD_ZIP = UPLOAD_SOURCE / "sersem_kaggle_source.zip"
PROJECT_REPO_URL = ""  # your framework repository, not the original SERSEM URL
PROJECT_REF = ""       # immutable framework commit SHA for github mode
PROJECT_ROOT = WORK_ROOT / "sersem-project-fresh-v1"
UPSTREAM_NAME = "SERSEM-Poisoned-Chalice-Competition-2026"
UPSTREAM_SHA = "413f56040e5b4805bcf15ed794dec56bc4e16b41"
UPSTREAM_URL = "https://github.com/Serdark4ra/SERSEM-Poisoned-Chalice-Competition-2026.git"
OUTPUT_ROOT = WORK_ROOT / "sersem_outputs"
MODE = "resource_validation"  # smoke | resource_validation | full; new RUN_ID per mode
RUN_ID = "starcoder2-3b-resource-validation-v1"
SEED = 42
INVALID_CONTENT_POLICY = "drop"  # "error" records diagnostics and stops instead
MODEL_ID = "bigcode/starcoder2-3b"
MODEL_REVISION = "main"  # metadata API resolves this to an immutable SHA below
DATASET_ID = "AISE-TUDelft/Poisoned-Chalice"
DATASET_REVISION = "main"
DTYPE = "float16"  # bfloat16 requires a compatible accelerator and a new run ID
FAITHFUL_MODE = True  # False required for any shorter context/reduced feature precision
MAX_LENGTH = 8192  # Faithful reference cap, including validation; smoke may explicitly use 512 with faithful=False
DEV_PER_LANGUAGE = 40 if MODE == "smoke" else 400 if MODE == "resource_validation" else 6250
TEST_PER_LANGUAGE = 20 if MODE == "smoke" else 200 if MODE == "resource_validation" else 5000
FEATURE_CHUNK_SIZE = 8
PROBE_MODE = "train"  # "verified" requires a trusted pickle + provenance manifest
PROBE_PATH = ""
TRUST_PROBE_PICKLE = False
BRANCHES = ["lumia", "greybox", "fusion"]  # ["greybox"] skips probe fitting
FUSION_MODE = "fixed"  # "fixed" or "fit_grid"; both are OUR experiments
FUSION_ALPHA = 0.5
FUSION_ALPHA_GRID = [i / 10 for i in range(11)]
GREYBOX_EPSILON = 1e-6
GREYBOX_LOGIT_CHUNK_SIZE = 64
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Resource controls; no implicit approximation. Explicit setup prepares caches first.
MODEL_CACHE_DIR = WORK_ROOT / "hf_cache" / "hub"  # can point to attached HF hub cache
MODEL_LOCAL_FILES_ONLY = True  # model loads offline after explicit AUTO_DOWNLOAD preparation
DATA_ACCESS = "cached"  # explicitly prepare missing Arrow cache below; or choose "streaming"
LAYER_GROUP_SIZE = 4  # 25k shadow/3B: ~2.29 GiB features; 8 extraction passes for 30 layers
FEATURE_STORAGE_DTYPE = "float32"  # float16 is lossy and forbidden in faithful mode
SCORE_CHUNK_SIZE = 32  # matches original probe inference batch size
TEXT_CACHE_GIB = 0.25  # bounded compressed selected source; set 0 for no source-text caching
DISK_RESERVE_GIB = 1.0
RAM_RESERVE_GIB = 3.0
SESSION_BUDGET_MINUTES = 480  # safe pause between chunks/layers; 0 disables this budget
LOG_INTERVAL_SECONDS = 30
RESOURCE_VALIDATION_EXAMPLES = 8  # longest shadow files by characters; actual token coverage reported
ATTENTION_BACKEND = "upstream"  # upstream | sdpa | eager; overrides forbidden in faithful mode
DISABLE_UNUSED_KV_CACHE = True  # no past states reused; changes allocation, not hidden/logit formulas
EVALUATE_HELDOUT = MODE != "resource_validation"  # validation mode does not expose held-out metrics by default
RESUME_FROM = ""  # e.g. /kaggle/input/my-checkpoints/<RUN_ID>; trusted own run directory only

# Timing does not alter cfg/attack identity or invalidate valid checkpoints.
RUN_TIMING_BENCHMARK = True
BENCHMARK_EXAMPLES = 30  # 20--50 training examples; explicit synthetic cap stress if needed
TIMING_ONLY = True  # first run: benchmark and STOP before probe training/evaluation

# Explicit fresh-session preparation (runs only on Kaggle, not during source checks).
AUTO_DOWNLOAD_MODEL_IF_MISSING = True
DOWNLOAD_MIN_FREE_GIB = 3.0
AUTO_PREPARE_DATASET_IF_MISSING = True
DATASET_PREPARATION_BUDGET_GIB = 8.0  # conservative peak allowance when metadata is incomplete
DATASET_CACHE_DIR = WORK_ROOT / "hf_cache/datasets"  # edit for an attached Arrow cache


## 2. Acquire and verify source


`uploaded`: attach `notebooks/sersem_kaggle_source.zip` as a private Kaggle Dataset. The allowlisted archive excludes `.env`, Git history, dataset contents and results. Both extracted folders and the zip are supported.

`github`: push this framework to your own GitHub repository **without the nested upstream checkout or any data/secrets**, set its URL and exact commit above. The cell clones your framework, then separately clones the pinned upstream with history. It never sources the upstream `.env`. Internet is still needed for dependencies and Hugging Face in either mode.


In [ ]:
import subprocess, shutil, zipfile, hashlib, json, sys, os
EXPECTED_FRAMEWORK_SHA256 = {'sersem_baseline.py': '9263371c56a5a1087f8a4e8d2adc9371f4461c1ffbf051c6fcee459c269e8a57', 'weighted_anomaly.py': '41aa9b87aa3dcfd86118c3618732f48a9a21e111d97e7e6322a41e8eef54573e', 'kaggle_setup.py': '990ad092df298db20ab9203d00b8bd5f108ab521154356405475f9595843cfb6', 'resource_data.py': 'ff737f86fa9084d5c72a9be45b5b24a54ac67748acbe45fcd89d7e9b96fd82d6', 'resource_pipeline.py': '5565efc812ffef8cd054b7dedc022dda0bdb1c493cd3a63815f3a5add3fbb4c8', 'research_execution.py': '348152e2f401cd0eaf997c604b8dba200eca43e3409159c2d6cc8bef7722e477', 'cache_setup.py': '560d6f2dd77530d2f461c0ad247f1fb7c2200336e6a37d99b25acde80bc7349a', 'requirements-kaggle.txt': '2dd3ad3196b4cdbba4d223df6b731e4f95b3ac468a216d84ce72eb3ead5e2d37', 'upstream_manifest.json': 'f8cb8621f0e29f1ac2864aa805449f21a59f528a9bcf12f34012a1f75ea0fbc1'}
# Prefer Kaggle's already-extracted source; never re-zip or copy datasets/weights.
if not (PROJECT_ROOT / "shared/sersem_baseline.py").exists():
    if PROJECT_ROOT.exists():
        raise FileExistsError(f"Existing incomplete project at {PROJECT_ROOT}; select a fresh PROJECT_ROOT")
    if SOURCE_MODE == "github":
        if not PROJECT_REPO_URL or not PROJECT_REF:
            raise ValueError("Set framework repository URL and immutable commit")
        subprocess.run(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", PROJECT_REF], check=True)
    elif SOURCE_MODE == "uploaded":
        if (UPLOAD_SOURCE / "shared/sersem_baseline.py").is_file():
            source_files = ["shared/" + name for name in EXPECTED_FRAMEWORK_SHA256]
            source_files += ["shared/author_source_search.json", "neighbourhood_attack/runner.py", "neighbourhood_attack/perturb.py", "neighbourhood_attack/README.md"]
            source_files += [UPSTREAM_NAME + "/" + name for name in ["README.md", "requirements.txt", "run_pipeline.sh", "code/ASTExtractor.py", "code/LinterExtractor.py", "code/LumiaAttack.py", "code/MIAttack.py", "code/plot_results.py", "code/run.py", "code/transformers_compat.py"]]
            for name in source_files:
                src = UPLOAD_SOURCE / name
                if not src.is_file():
                    raise FileNotFoundError(f"Extracted source is incomplete/outdated: {src}. Upload the rebuilt research source package.")
            PROJECT_ROOT.mkdir(parents=True)
            for name in source_files:
                dest = PROJECT_ROOT / name
                dest.parent.mkdir(parents=True,exist_ok=True)
                shutil.copy2(UPLOAD_SOURCE / name,dest)
            for name in ["README.md","AUDIT.md","KAGGLE_RESOURCES.md","RESEARCH_RUN.md"]:
                if (UPLOAD_SOURCE / name).is_file():
                    shutil.copy2(UPLOAD_SOURCE / name,PROJECT_ROOT / name)
            print("Copied allowlisted extracted source:",UPLOAD_SOURCE)
        elif UPLOAD_ZIP.is_file():
            with zipfile.ZipFile(UPLOAD_ZIP) as archive:
                for item in archive.infolist():
                    dest=(PROJECT_ROOT / item.filename).resolve()
                    if not dest.is_relative_to(PROJECT_ROOT.resolve()) or any(part in {".env",".git"} for part in Path(item.filename).parts):
                        raise ValueError("Unsafe archive path")
                archive.extractall(PROJECT_ROOT)
            print("Extracted source ZIP:",UPLOAD_ZIP)
        else:
            raise FileNotFoundError(f"Neither extracted source at {UPLOAD_SOURCE} nor ZIP at {UPLOAD_ZIP}. Check your Kaggle Dataset mount.")
    else:
        raise ValueError("Unknown SOURCE_MODE")
# Bind the upstream provenance manifest itself before reading its contents.
for name, expected_hash in EXPECTED_FRAMEWORK_SHA256.items():
    helper_path=PROJECT_ROOT / "shared" / name
    if not helper_path.is_file() or hashlib.sha256(helper_path.read_bytes()).hexdigest()!=expected_hash:
        raise RuntimeError(f"Outdated framework file: {name}. Update the source Dataset and choose a fresh PROJECT_ROOT; existing source was not overwritten.")
UPSTREAM=PROJECT_ROOT / UPSTREAM_NAME
if not (UPSTREAM / "code/LumiaAttack.py").exists():
    if SOURCE_MODE != "github":
        raise FileNotFoundError("Uploaded package lacks the pinned upstream sources")
    subprocess.run(["git","clone",UPSTREAM_URL,str(UPSTREAM)],check=True)
    subprocess.run(["git","-C",str(UPSTREAM),"checkout","--detach",UPSTREAM_SHA],check=True)
manifest=json.loads((PROJECT_ROOT / "shared/upstream_manifest.json").read_text())
if manifest["commit_sha"]!=UPSTREAM_SHA:
    raise RuntimeError("Unexpected upstream provenance commit")
if (UPSTREAM / ".git").exists():
    actual=subprocess.check_output(["git","-C",str(UPSTREAM),"rev-parse","HEAD"],text=True).strip()
    if actual!=UPSTREAM_SHA:
        raise RuntimeError("Unexpected upstream HEAD")
for name, expected in manifest["source_sha256"].items():
    if hashlib.sha256((UPSTREAM / name).read_bytes()).hexdigest()!=expected:
        raise RuntimeError(f"Upstream source changed: {name}")
print("Verified pinned upstream source:",UPSTREAM_SHA)
# Verify standalone neighbourhood source as well.
EXPECTED_NEIGHBOURHOOD_SHA256 = {'neighbourhood_attack/runner.py': '7c62ed01c13faaeeb4fa52e78252cb18a98bae197921c9e32391244a1c409cc2', 'neighbourhood_attack/perturb.py': '031d7b66011f0b9711639eb130dc2439badc525b97d8dea1799bb8d68c56f706'}
for relative_name, expected_hash in EXPECTED_NEIGHBOURHOOD_SHA256.items():
    if hashlib.sha256((PROJECT_ROOT / relative_name).read_bytes()).hexdigest() != expected_hash:
        raise RuntimeError("Outdated neighbourhood helper; use refreshed source and a fresh PROJECT_ROOT")
# End neighbourhood source verification.
sys.path.insert(0, str(PROJECT_ROOT))
for module_name in ["shared.kaggle_setup","shared.sersem_baseline","shared.weighted_anomaly","shared.resource_data","shared.resource_pipeline","shared.research_execution","shared.cache_setup","neighbourhood_attack.runner","neighbourhood_attack.perturb"]:
    loaded_module=sys.modules.get(module_name)
    if loaded_module is not None and not Path(loaded_module.__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()):
        raise RuntimeError("Helpers from another project are loaded; restart the kernel and run in order")


## 3. Stable Kaggle Python 3.13 dependency setup

Run after source acquisition and before importing numerical/model libraries. The installer keeps compatible installed NumPy, Pandas, Matplotlib, PyArrow, SciPy, scikit-learn and the CUDA stack by writing pip constraints to their **actual installed versions**. Requirements use supported ranges for those libraries; the attack API dependencies retain explicit pins. If an installed protected library is outside its supported range, setup stops without replacing it. Use a compatible fresh Kaggle image in that case.

Pip runs only when requirements are missing or mismatched, with no forced reinstall or blanket upgrade; rerunning a successful setup skips pip. System Enchant/dictionary packages are installed only when missing. If a changed dependency was already imported by the kernel, restart **once**, then rerun from the first cell. The next setup should be a no-op. Do not repeatedly run pip in a stale kernel or upgrade dependencies just to silence unrelated Kaggle package warnings.

Internet, PyPI wheel availability, apt permissions and compatibility of the current Kaggle image remain runtime requirements. Python 3.11–3.13 is supported by the setup policy; this notebook targets the reported Python 3.13 runtime. Installation and imports do not run any model inference.

The installer selects protected versions using `importlib.metadata.version(name)`, matching its requirement checks. It does not take the last entry from all distribution metadata: Kaggle can retain duplicate entries (for example NumPy 1.26 and 2.x). Generated constraints are checked against requirements before pip. The selected and duplicate metadata are saved to `/kaggle/working/sersem_dependency_diagnostics.json`; pip failures retain the full explanation in `sersem_pip_install.log`.


In [ ]:
# 3. Install once, keeping Kaggle's compatible numerical/CUDA stack intact.
import sys, os, subprocess
from pathlib import Path
from shared.kaggle_setup import install_dependencies
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"  # avoid optional transport chunk-cache duplication
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
requirements_file = PROJECT_ROOT / "shared/requirements-kaggle.txt"
# Check system packages without importing Enchant or any numerical library.
system_packages = ["libenchant-2-2", "hunspell-en-us"]
system_status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *system_packages],
                               capture_output=True, text=True)
if system_status.returncode != 0 or system_status.stdout.splitlines().count("install ok installed") != len(system_packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *system_packages], check=True)
installation_report = install_dependencies(requirements_file, WORK_ROOT)
print("Dependencies ready. Proceed to the initialization cell.")
# Optional authentication: uncomment if needed; never print/save the token.
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")


## 4. Initialize every helper import, cfg and OUT

This complete cell imports **all** functions used later, checks the environment/GPU and settings, resolves immutable Hugging Face metadata revisions, seeds Python/NumPy/PyTorch, creates the run directory and persists its configuration plus actual installed dependency versions. Its successful `Initialized cfg and OUT` message is required before data loading. It does not download model weights or instantiate a model. Do not replace this cell with a shorter import-only cell.

For a new run leave `RESUME_FROM=""`. For another Kaggle session, attach a trusted saved run directory as a Dataset, keep its RUN_ID and settings, and set `RESUME_FROM` to that directory. Its pinned HF revisions are reused, so a newer `main` does not silently invalidate the checkpoint. Mathematical settings, code hashes, sample order and actual dependency versions must match. Cache paths, data access mode, logging interval, storage/RAM reserves, text-cache budget and session time budget may change; they are recorded but excluded from mathematical checkpoint identity. Source/probe feature precision and layer grouping may not change mid-run. Do not reuse earlier v5 smoke checkpoints as resource-mode checkpoints.


In [ ]:
# 4. Complete initialization — this cell MUST succeed before dataset manifest creation.
import sys, os, json, hashlib, random, math
from pathlib import Path
from shared.kaggle_setup import validate_environment
from shared.research_execution import (benchmark_sersem, install_probe_timer, estimate_sersem_runtime, score_with_milestones as score_branches)
from shared.resource_data import SourceStore
from shared.resource_pipeline import (
    Monitor, preflight, restore_run, semantic_cfg, attach_runtime,
    validate_resources,
)
from shared.sersem_baseline import build_attack, prepare_probes, write_json
from shared.weighted_anomaly import (
    WeightedAnomalyMIA, split_calibration,
    fit_fusion, add_fusion, freeze_thresholds, evaluate_branches,
)
environment_report = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt")
import numpy as np
import torch
from huggingface_hub import HfApi
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator. CPU inference is forbidden.")
if INVALID_CONTENT_POLICY not in {"drop", "error"}:
    raise ValueError("INVALID_CONTENT_POLICY must be drop or error")
if MODE not in {"smoke", "resource_validation", "full"} or DTYPE not in {"float16", "bfloat16"}:
    raise ValueError("Invalid mode or dtype")
if not BRANCHES or len(BRANCHES) != len(set(BRANCHES)) or not set(BRANCHES) <= {"lumia", "greybox", "fusion"}:
    raise ValueError("Invalid branches")
if PROBE_MODE not in {"train", "verified"} or (PROBE_MODE == "verified" and (not PROBE_PATH or not TRUST_PROBE_PICKLE)):
    raise ValueError("Verified probes require PROBE_PATH and explicit provenance trust")
if not isinstance(BENCHMARK_EXAMPLES, int) or not 20 <= BENCHMARK_EXAMPLES <= 50:
    raise ValueError("BENCHMARK_EXAMPLES must be 20..50")
if not isinstance(SEED, int) or not 0 <= SEED < 2**32:
    raise ValueError("SEED must be an integer in [0, 2**32)")
if not all(isinstance(value, int) for value in [MAX_LENGTH, DEV_PER_LANGUAGE, TEST_PER_LANGUAGE, FEATURE_CHUNK_SIZE, GREYBOX_LOGIT_CHUNK_SIZE]):
    raise ValueError("Lengths, sampling counts and chunk sizes must be integers")
if MAX_LENGTH < 2 or DEV_PER_LANGUAGE < 40 or DEV_PER_LANGUAGE % 2 or TEST_PER_LANGUAGE < 4 or TEST_PER_LANGUAGE % 2 or min(FEATURE_CHUNK_SIZE, GREYBOX_LOGIT_CHUNK_SIZE) < 1:
    raise ValueError("Invalid context/chunk size or insufficient balanced calibration/evaluation samples")
if not math.isfinite(GREYBOX_EPSILON) or GREYBOX_EPSILON <= 0:
    raise ValueError("GREYBOX_EPSILON must be positive and finite")
if FUSION_MODE not in {"fixed", "fit_grid"} or not math.isfinite(FUSION_ALPHA) or not 0 <= FUSION_ALPHA <= 1:
    raise ValueError("Invalid fusion settings")
if FUSION_MODE == "fit_grid" and (not FUSION_ALPHA_GRID or any(not math.isfinite(a) or not 0 <= a <= 1 for a in FUSION_ALPHA_GRID)):
    raise ValueError("Fusion grid must be nonempty and within [0,1]")
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("RUN_ID must be one directory name")
if DATA_ACCESS not in {"cached", "streaming"} or ATTENTION_BACKEND not in {"upstream", "sdpa", "eager"}:
    raise ValueError("Invalid resource data access/attention mode")
if FEATURE_STORAGE_DTYPE not in {"float32", "float16"}:
    raise ValueError("Invalid feature storage dtype")
if not all(isinstance(v, int) and v > 0 for v in [LAYER_GROUP_SIZE, SCORE_CHUNK_SIZE, RESOURCE_VALIDATION_EXAMPLES]):
    raise ValueError("Layer/score/validation sizes must be positive integers")
if not all(math.isfinite(v) and v >= 0 for v in [TEXT_CACHE_GIB, SESSION_BUDGET_MINUTES]) or min(DISK_RESERVE_GIB, RAM_RESERVE_GIB, LOG_INTERVAL_SECONDS) <= 0:
    raise ValueError("Invalid resource reserves/budget")
if FAITHFUL_MODE and (MAX_LENGTH != 8192 or DTYPE != "float16" or FEATURE_STORAGE_DTYPE != "float32" or ATTENTION_BACKEND != "upstream"):
    raise ValueError("Faithful mode forbids shorter context, reduced feature precision or changed attention selection")
OUT = OUTPUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM:
    restore_run(RESUME_FROM, OUT)
# On resume, reuse already pinned SHAs instead of silently resolving a new main.
if (OUT / "metrics.json").exists():
    raise RuntimeError("Held-out evaluation already completed. Read saved metrics; use a new run only for a prespecified experiment.")
saved_config_path = OUT / "config.json"
saved_cfg = json.loads(saved_config_path.read_text()) if saved_config_path.exists() else None
# Metadata requests only: these do not download model weights.
api = HfApi()
model_revision = (saved_cfg["model_revision"] if saved_cfg and saved_cfg["model_id"] == MODEL_ID and saved_cfg["requested_model_revision"] == MODEL_REVISION
                  else api.model_info(MODEL_ID, revision=MODEL_REVISION).sha)
dataset_revision = (saved_cfg["dataset_revision"] if saved_cfg and saved_cfg["dataset_id"] == DATASET_ID and saved_cfg["requested_dataset_revision"] == DATASET_REVISION
                    else api.dataset_info(DATASET_ID, revision=DATASET_REVISION).sha)
if not model_revision or not dataset_revision:
    raise RuntimeError("Hugging Face did not return immutable model/dataset revisions")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Every cfg key consumed by either Stage 1 helper is initialized here.
cfg = dict(
    model_id=MODEL_ID, model_revision=model_revision,
    dataset_id=DATASET_ID, dataset_revision=dataset_revision,
    requested_model_revision=MODEL_REVISION, requested_dataset_revision=DATASET_REVISION,
    dtype=DTYPE, seed=SEED, mode=MODE, upstream_sha=UPSTREAM_SHA,
    invalid_content_policy=INVALID_CONTENT_POLICY,
    runner_sha256=hashlib.sha256(b"".join((PROJECT_ROOT / "shared" / name).read_bytes()
                                        for name in ["sersem_baseline.py", "weighted_anomaly.py", "kaggle_setup.py", "resource_pipeline.py", "resource_data.py"])).hexdigest(),
    max_length=MAX_LENGTH, dev_per_language=DEV_PER_LANGUAGE,
    test_per_language=TEST_PER_LANGUAGE, feature_chunk_size=FEATURE_CHUNK_SIZE,
    probe_mode=PROBE_MODE, probe_path=PROBE_PATH, trust_probe_pickle=TRUST_PROBE_PICKLE,
    branches=list(BRANCHES),
    greybox=dict(implementation="section2.1-reconstruction-v1", epsilon=GREYBOX_EPSILON,
                 logit_chunk_size=GREYBOX_LOGIT_CHUNK_SIZE),
    fusion=dict(mode=FUSION_MODE, alpha=FUSION_ALPHA, alpha_grid=list(FUSION_ALPHA_GRID)),
    versions=environment_report["versions"], python_version=environment_report["python"],
    resources=dict(enabled=True, faithful=FAITHFUL_MODE, model_cache_dir=str(MODEL_CACHE_DIR),
                   model_local_files_only=MODEL_LOCAL_FILES_ONLY, data_access=DATA_ACCESS,
                   layer_group_size=LAYER_GROUP_SIZE, feature_dtype=FEATURE_STORAGE_DTYPE,
                   score_chunk_size=SCORE_CHUNK_SIZE, text_cache_gib=TEXT_CACHE_GIB,
                   disk_reserve_gib=DISK_RESERVE_GIB, ram_reserve_gib=RAM_RESERVE_GIB,
                   session_budget_minutes=SESSION_BUDGET_MINUTES, log_interval_seconds=LOG_INTERVAL_SECONDS,
                   validation_examples=RESOURCE_VALIDATION_EXAMPLES, attention_backend=ATTENTION_BACKEND,
                   disable_kv_cache=DISABLE_UNUSED_KV_CACHE),
)
OUT = OUTPUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
config_path = OUT / "config.json"
if config_path.exists() and semantic_cfg(json.loads(config_path.read_text())) != semantic_cfg(cfg):
    raise RuntimeError("Configuration, revisions, helpers or actual dependency versions changed. Select a NEW RUN_ID; existing outputs were preserved.")
write_json(config_path, cfg)
write_json(OUT / "environment.json", environment_report)
print("Initialized cfg and OUT:", config_path)
print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda)
print(json.dumps(cfg, indent=2))

monitor = Monitor(cfg, OUT)
monitor.tick("initialized", 0, force=True)
monitor.guard()


## Explicit cache preparation
Reuse a complete pinned snapshot, otherwise download only required model/tokenizer files when explicitly enabled. Cached data mode may prepare missing Arrow files; streaming is a separate explicit choice. At least 3 GiB is reserved for working files after setup; metadata/budget checks can fail before download. These are setup operations, not inference.


In [ ]:
from shared.cache_setup import ensure_model_snapshot, ensure_dataset_cache
if DOWNLOAD_MIN_FREE_GIB < 3:
    raise ValueError("Keep at least 3 GiB free after asset preparation")
# cfg already contains immutable SHAs; runtime model loading stays local-only.
write_json(OUT / "cache_setup_settings.json",dict(auto_download_model=AUTO_DOWNLOAD_MODEL_IF_MISSING,
           auto_prepare_dataset=AUTO_PREPARE_DATASET_IF_MISSING,min_free_gib=DOWNLOAD_MIN_FREE_GIB,
           dataset_preparation_budget_gib=DATASET_PREPARATION_BUDGET_GIB,data_access=DATA_ACCESS))
model_cache_report = ensure_model_snapshot(cfg["model_id"],cfg["model_revision"],MODEL_CACHE_DIR,
                     AUTO_DOWNLOAD_MODEL_IF_MISSING,DOWNLOAD_MIN_FREE_GIB,WORK_ROOT,OUT / "model_cache_setup.json")
dataset_cache_report = ensure_dataset_cache(cfg,AUTO_PREPARE_DATASET_IF_MISSING,
                       DATASET_PREPARATION_BUDGET_GIB,DOWNLOAD_MIN_FREE_GIB,WORK_ROOT,OUT / "dataset_cache_setup.json")
print("Asset setup ready. Next: source manifests/resource preflight, then load target and benchmark.")


## Metadata-only ingestion and resource preflight

`SourceStore` scans official train/test splits into a compact SQLite hash/ID/label index. It reads source text one row at a time; it never converts whole splits to Pandas or copies the complete source corpus into outputs. `DATA_ACCESS="cached"` uses existing Arrow caches via `builder.as_dataset()` without preparing a second dataset. If caches are absent it fails; explicitly select `"streaming"` to iterate remotely without a full download. Text retrieval then loads only selected IDs in bounded batches. A capped compressed text cache avoids some repeat scans; zero disables caching. The metadata index still scales with corpus row count, and the selected sample manifests/scalar metrics fit in RAM.

All valid source hashes from the entire official test split are excluded from development before global keep-first deduplication. Class/language sampling uses the original configured seeded Pandas sampling order on metadata. The same source-row IDs/content hashes drive every branch and every layer group. Unknown labels, conflicting labels, insufficient valid samples or content/hash changes fail explicitly. Null/non-string/blank exclusions are recorded in `excluded_content_rows.csv` and `data_quality.json`; accepted source is never rewritten.

| Mode | Shadow | Fusion fitting | Threshold calibration | Held-out pool | Default held-out evaluation |
|---|---:|---:|---:|---:|---|
| smoke | 160 | 20 | 20 | 100 | yes |
| resource_validation | 1,600 | 200 | 200 | 1,000 | no |
| full | 25,000 | 3,125 | 3,125 | 25,000 | yes |

Set mode first, then use a new RUN_ID. Faithful mode keeps the upstream **8192-token cap**, fp16 model and float32 pooled features even for the intermediate run. A 512-token smoke run requires explicit `FAITHFUL_MODE=False`. Counts are balanced per language/class before calibration splitting; odd calibration stratum sizes may differ by one. Full mode is the existing paper-scale protocol, not all ~650k official rows.

Before model loading, preflight records current GPU/host/disk capacity, cached weight bytes, dataset/cache/checkpoint footprints and analytical activation/logit/feature estimates. With 5.3 GiB free, model weights must already exist or be attached separately. `MODEL_LOCAL_FILES_ONLY=True` is default; no automatic model download or cache eviction occurs. A four-layer float32 group at 25k rows occupies ~2.29 GiB, versus ~17.17 GiB for all 30 layers. Estimated reserves and working files must also fit. Preflight cannot predict attention workspace peaks; the next cell performs a real manual GPU stress check on the longest selected shadow sources, reporting actual token coverage.


In [ ]:
source_store = SourceStore(cfg, OUT, monitor)
shadow, calibration, heldout = source_store.manifests()
for name, frame in [("shadow", shadow), ("calibration", calibration), ("heldout_test", heldout)]:
    print(name, frame.groupby(["language", "is_member"]).size().to_dict())
    frame.drop(columns=["content"], errors="ignore").to_parquet(OUT / f"{name}_manifest.parquet", index=False)

fusion_fit, threshold_calibration = split_calibration(calibration, cfg["seed"], OUT)

print("Input exclusions:", OUT / "excluded_content_rows.csv")
print(json.dumps(json.loads((OUT / "data_quality.json").read_text()), indent=2))

resource_plan = preflight(cfg, OUT, monitor, shadow_rows=len(shadow))


## Bounded extraction, original probe training and resource validation

The next cell loads one GPU model/tokenizer and validates real shadow examples only when **you run it on Kaggle**. It records GPU allocated/reserved/peak memory, available host RAM/disk, sample/token throughput and whether the selected long sources actually exercise the configured token cap. Lack of full-cap coverage is not a success guarantee. An OOM is an explicit failure, never a context/precision fallback.

LUMIA still calls the unchanged upstream tokenizer, structural weight mask and hooked weighted+mean pooling. The original language omission and shallow early-stop snapshot behavior remain. Upstream extraction is invoked one source at a time so its final logits/unused outputs are released before the next example. Unused KV-cache outputs are disabled by default without changing the causal hidden/logit formulas. No model quantization, autocast conversion or alternative scoring algorithm is introduced.

Only `LAYER_GROUP_SIZE` layers' pooled features persist at once as float32 memory maps. Completion flags commit after feature flushes. The unchanged upstream `_train_probe` receives its full original per-layer training/early-stop arrays, with the same Adam/BCE/minibatches/epochs/stopping behavior. The original label-stratified 60/20/20 indices and top-five selection rule are reproduced outside the source checkout. Completed per-layer states, selection AUCs and Python/NumPy/PyTorch/CUDA RNG states are checkpointed atomically. Model/source I/O RNG effects are restored before fitting. A partially trained probe restarts from its last completed-layer boundary; it does **not** resume mid-epoch. After all layers in a group are committed, only that run's temporary feature files are removed. No user dataset/model cache is deleted.

Four-layer groups require **8 shadow extraction passes** for 30 layers. Each pass computes all transformer layers, retaining only the current group. This is the deliberate disk-versus-time tradeoff. At 25k shadow samples, it means up to 200k training-extraction forwards before the two scoring branches. Larger groups need more disk; smaller groups require more passes. A full run is not assumed to fit a single Kaggle session. Throughput/ETA logs should determine whether continuing is practical.

Grey-box math is unchanged: raw causal target logit → uniform full-vocabulary population Z-score in float32 → sigmoid → sum-normalized structural weighted mean. Target offsets, exclusions, epsilon, language-aware mask and all documented reconstruction assumptions remain those in AUDIT.md. A separate forward still allocates full model logits, but vocabulary reductions use the existing fixed-size float32 chunks and no activations/logits are saved. Per-sample grey scores and per-chunk LUMIA results are stored in a compact SQLite ledger rather than thousands of JSON/feature files. Probe outputs remain the original top-five arithmetic mean.

`FEATURE_STORAGE_DTYPE="float16"`, shorter context, bf16 or an attention-backend override are explicit approximations/changes, rejected by faithful mode. GPU batching/grouping can produce backend-dependent last-bit differences; mathematical preservation is not a claim of bitwise equality or published-result reproduction. No numerical equivalence test was run locally.


In [ ]:
# Reuse the same live target when continuing after the timing-only pass.
attack = globals().get("attack")
if attack is None:
    attack = build_attack(cfg, UPSTREAM, OUT)
    attack._research_identity = semantic_cfg(cfg)
elif getattr(attack, "_research_identity", None) != semantic_cfg(cfg):
    raise RuntimeError("A different target/config is loaded. Restart before changing experiments.")
greybox = WeightedAnomalyMIA(attack.model, attack.tokenizer, cfg, OUT) if set(cfg["branches"]) & {"greybox", "fusion"} else None
attach_runtime(attack, cfg, OUT, source_store, monitor, greybox)
install_probe_timer(attack, OUT)
if RUN_TIMING_BENCHMARK:
    timing_report = benchmark_sersem(attack, greybox, source_store, shadow, cfg, OUT, count=BENCHMARK_EXAMPLES)
    print(json.dumps(timing_report, indent=2))
    runtime_forecast = estimate_sersem_runtime(cfg, OUT, len(shadow), len(fusion_fit), len(threshold_calibration), len(heldout) if EVALUATE_HELDOUT else 0)
if TIMING_ONLY:
    print("Timing-only pass finished. No probes trained or held-out scores generated. Inspect measured timing_samples.csv; set TIMING_ONLY=False and rerun this cell and evaluation to continue with this loaded model.")
else:
    resource_validation = validate_resources(attack, greybox, shadow, cfg)
    if set(cfg["branches"]) & {"lumia", "fusion"}:
        probe_path = prepare_probes(attack, cfg, shadow, calibration, heldout, OUT)
        shadow.to_parquet(OUT / "shadow_manifest.parquet", index=False)
        print("Original probes ready:", probe_path, "selected layers:", attack.ensemble_layers)
    runtime_forecast = estimate_sersem_runtime(cfg, OUT, len(shadow), len(fusion_fit), len(threshold_calibration), len(heldout) if EVALUATE_HELDOUT else 0)


## Freeze the fusion experiment and thresholds, then evaluate once

`fixed` uses the prespecified alpha without label fitting. `fit_grid` maximizes AUC on **fusion_fit only**, with ties resolved by proximity to the prespecified alpha and then smaller alpha. Raw component scores are mixed without scaling, intercept fitting, per-language tuning or test-set score-direction selection. Neither the mixture nor its weight selection is claimed to be the authors' method. Component scores are reported even when fusion picks an endpoint.

The chosen mixture is frozen in `fusion.json` before threshold calibration. Separate per-branch thresholds at empirical calibration FPR <=1% are frozen in `thresholds.json` **before held-out scoring**. Held-out ROC TPR at FPR <=1% (no interpolation) is descriptive and distinct from operating TPR/FPR at the frozen thresholds. Do not tune on this held-out result. A completed run cannot be evaluated again in the same directory.

Scoring now reads bounded source chunks and writes resumable scalar/vector records keyed by sample identity, mathematical configuration and actual probe checksum. It retains no whole-test activation array. Fusion fitting and threshold selection still use separate development partitions and freeze before held-out scoring. Resource-validation mode stops after calibration by default (`EVALUATE_HELDOUT=False`), so capacity decisions need not inspect test metrics. Do not use any intermediate held-out AUC, if explicitly enabled, for choosing the final attack parameters.


In [ ]:
if TIMING_ONLY:
    print("Evaluation skipped in timing-only mode.")
else:
    if (OUT / "metrics.json").exists():
        raise RuntimeError("Held-out evaluation already completed. Read saved metrics; do not tune on it.")
    if "fusion" in cfg["branches"]:
        fusion_fit_scored = score_branches(attack, greybox, fusion_fit, cfg, OUT / "fusion_fit_scores.parquet")
    else:
        fusion_fit_scored = fusion_fit  # no fusion means no unnecessary fitting-pool inference
    fusion_state = fit_fusion(fusion_fit_scored, cfg, OUT)
    calibration_scored = score_branches(attack, greybox, threshold_calibration, cfg, OUT / "threshold_calibration_scores.parquet")
    calibration_scored = add_fusion(calibration_scored, fusion_state)
    calibration_scored.drop(columns=["content"], errors="ignore").to_parquet(OUT / "threshold_calibration_scores.parquet", index=False)
    frozen_thresholds = freeze_thresholds(calibration_scored, cfg, OUT)
    if EVALUATE_HELDOUT:
        # All fitting/selection is finished. Held-out labels enter only the final report.
        heldout_scored = score_branches(attack, greybox, heldout, cfg, OUT / "heldout_test_scores.parquet")
        heldout_scored = add_fusion(heldout_scored, fusion_state)
        report = evaluate_branches(heldout_scored, cfg, OUT)
        report["runtime"] = json.loads((OUT / "heldout_runtime.json").read_text())
        report["runtime"]["observed_session_since_initialization_seconds"] = __import__("time").monotonic() - monitor.started
        write_json(OUT / "metrics.json", report)
        print(json.dumps(report, indent=2))
    else:
        write_json(OUT / "resource_validation_complete.json", {"heldout_evaluated": False, "mode": MODE})
        print("Intermediate resource validation finished without held-out evaluation. Inspect resource logs/ETA before a separate full run.")


## Persist outputs and resume across Kaggle sessions

Outputs live under `/kaggle/working/sersem_outputs/<RUN_ID>/`. This is **ephemeral local storage**, not an external backup. SQLite/memmap checkpoints provide process/session restart only if you persist the directory before it disappears.

1. Let the cell finish or reach `ResourcePause` between chunks/layers. Do not snapshot files while another cell is writing. The time budget is checked at chunk/layer boundaries; the unchanged `_train_probe` can run past the budget until its current layer finishes.
2. Save a Kaggle notebook version **including outputs**, or create a private Kaggle Dataset from the complete run directory, or download it to durable storage. No automatic upload/publication is performed. Avoid making a second multi-GiB ZIP in the remaining working disk; preserve the directory/files directly.
3. Include `config.json`, environment/manifests, `source_index.sqlite`, `scores.sqlite`, `probe_training.pkl`, `probes/` and the **entire** `active_features/` directory if a group is in progress. The bounded `selected_text.sqlite` source cache is optional; omitting it causes source rereads. Resource logs, preflight/validation reports and calibration/metric outputs should also be retained. Own-run pickles are trusted executable artifacts: never load an untrusted snapshot.
4. Next session, attach that saved directory as a Kaggle Dataset, select the same code/package environment and mathematical settings/RUN_ID, and set `RESUME_FROM` to its mount. Restore copies into a writable run directory without overwriting different files. Empty/fresh working space must fit the active group plus reserve. The pinned revisions are reused.
5. Model weights and the original full dataset caches are **not** in the run checkpoint. Preserve/attach them separately, set `MODEL_CACHE_DIR` to the matching HF hub-cache root, and use `DATA_ACCESS="streaming"` explicitly if the Arrow cache is unavailable. With only 5.3 GiB free, redownloading a missing multi-GiB model is generally infeasible. Never delete caches blindly.

Reports include `resource_preflight.json`, `loaded_model_resources.json`, `resource_validation.json` and `resource_log.jsonl` with live throughput/memory; source quality and role manifests; probe/RNG checkpoints; scalar/component/vector scores; frozen mixture/thresholds and per-branch final metrics/distributions. Source text is absent from score tables. Only the optional, capped text cache contains selected source content. No raw hidden activations/logits are persisted.

Full-size float32 features no longer need ~17 GiB simultaneously, but four-layer recomputation can cost many sessions. Per-forward 8192-token VRAM remains the main uncertainty on a 14.56 GiB T4. The entire official training corpus is not the default full experiment; if sample counts are greatly increased, even one exact per-layer matrix or upstream's full GPU train/validation tensor may exceed resources, and preflight must stop. More storage/VRAM or explicitly nonfaithful settings would then be required. No Stage 2 attack is implemented.
